In [8]:
import cv2
import numpy as np
from rknnlite.api import RKNNLite

# COCO 17개 관절 연결 정보 (어깨, 팔꿈치, 무릎 등 연결선)
SKELETON =  [[16, 14], [14, 12], [17, 15], [15, 13], [12, 13], 
             [6, 12], [7, 13], [6, 7], [6, 8], [7, 9], 
             [8, 10], [9, 11], [2, 3], [1, 2], [1, 3], 
             [2, 4], [3, 5], [4, 6], [5, 7]]

# 관절 및 뼈대 색상 (BGR)
POINT_COLOR = (0, 255, 0)   # 초록색
LINE_COLOR = (255, 0, 0)    # 파란색
BOX_COLOR = (0, 165, 255)   # 주황색

def draw_pose(image, objects, kpt_thresh=0.3):
    """
    image: cv2.imread()로 읽은 원본 이미지 (수정본 반환)
    objects: infer() 결과 리스트
    kpt_thresh: 표시할 관절의 신뢰도(Confidence) 임계값
    """
    img_draw = image.copy()

    for obj in objects:
        # 1. Bounding Box 그리기
        x1, y1, x2, y2 = obj["box"]
        cv2.rectangle(img_draw, (x1, y1), (x2, y2), BOX_COLOR, 2)
        cv2.putText(img_draw, f"{obj['score']:.2f}", (x1, max(0, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, BOX_COLOR, 1)

        keypoints = obj["keypoints"]  # [[x, y, conf], ...] (17개)

        # 2. 관절 간 뼈대(Skeleton) 그리기
        for p1_idx, p2_idx in SKELETON:
            # 1-indexed -> 0-indexed 변환
            kpt1 = keypoints[p1_idx - 1]
            kpt2 = keypoints[p2_idx - 1]

            x1_kpt, y1_kpt, conf1 = kpt1
            x2_kpt, y2_kpt, conf2 = kpt2

            # 두 관절 모두 신뢰도가 임계값 이상일 때만 선 연결
            if conf1 > kpt_thresh and conf2 > kpt_thresh:
                pt1 = (int(x1_kpt), int(y1_kpt))
                pt2 = (int(x2_kpt), int(y2_kpt))
                cv2.line(img_draw, pt1, pt2, LINE_COLOR, 2)

        # 3. 관절 점(Keypoints) 그리기
        for kpt in keypoints:
            x_kpt, y_kpt, conf = kpt
            if conf > kpt_thresh:
                cv2.circle(img_draw, (int(x_kpt), int(y_kpt)), 4, POINT_COLOR, -1)

    return img_draw

import cv2
import numpy as np
from rknnlite.api import RKNNLite

class YOLOv8PoseRKNN:
    def __init__(self, model_path, obj_thresh=0.5):
        self.obj_thresh = obj_thresh
        self.rknn = RKNNLite()
        
        # 1. 모델 로드 및 런타임 초기화
        ret = self.rknn.load_rknn(model_path)
        if ret != 0:
            raise RuntimeError("RKNN 모델 로드 실패")
            
        ret = self.rknn.init_runtime(core_mask=RKNNLite.NPU_CORE_0)
        if ret != 0:
            raise RuntimeError("RKNN 런타임 초기화 실패")

    def letterbox(self, im, new_shape=(320, 320), color=(0, 0, 0)):
        """dw, dh를 리턴하지 않는 letterbox 함수"""
        shape = im.shape[:2]  # [h, w]
        if isinstance(new_shape, int):
            new_shape = (new_shape, new_shape)

        ratio = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
        new_unpad = int(round(shape[1] * ratio)), int(round(shape[0] * ratio))

        dw = (new_shape[1] - new_unpad[0]) / 2
        dh = (new_shape[0] - new_unpad[1]) / 2

        if shape[::-1] != new_unpad:
            im = cv2.resize(im, new_unpad, interpolation=cv2.INTER_LINEAR)

        top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
        left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
        im = cv2.copyMakeBorder(im, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)

        return im, ratio

    def preprocess(self, img_path):
        """전처리: BGR/RGB 변환 및 320x320 입력 데이터 준비"""
        src_img = cv2.imread(img_path)
        
        # dw, dh 없이 letterbox 수행
        input_img, ratio = self.letterbox(src_img, new_shape=(320, 320))
        
        # NPU 입력 형식 맞춤 (RGB, NHWC)
        input_img = cv2.cvtColor(input_img, cv2.COLOR_BGR2RGB)
        input_data = np.expand_dims(input_img, axis=0)  # (1, 320, 320, 3)
        
        return src_img, input_data, ratio

    def infer(self, img_path):
        # 2. 이미지 전처리
        src_img, input_data, ratio = self.preprocess(img_path)
        
        # 3. NPU 추론 실행
        outputs = self.rknn.inference(inputs=[input_data])
        
        # 4. 후처리 진행 (dw, dh 전달 없음)
        objects = self._postprocess(outputs, src_img.shape[:2], (320, 320), ratio)
        return objects

    def _postprocess(self, outputs, src_shape, dst_shape, ratio):
        """후처리: ratio와 dst_shape(320x320)만으로 dw, dh 역산 후 복원"""
        box_outputs = outputs[:3]
        kpt_output = outputs[3][0]       # (17, 3, 2100)

        # ratio 및 원본/입력 크기 기반 dw, dh 역산
        src_h, src_w = src_shape[:2]
        dst_h, dst_w = dst_shape[:2]
        dw = (dst_w - src_w * ratio) / 2
        dh = (dst_h - src_h * ratio) / 2
        
        strides = [8, 16, 32]
        all_boxes, all_scores, kpt_indices = [], [], []

        anchor_count = 0
        for i, output in enumerate(box_outputs):
            pred = output[0].reshape(5, -1)  # (5, H*W)
            h, w = output.shape[2], output.shape[3]
            stride = strides[i]

            # Anchor Point 생성
            y = np.arange(h) * stride + stride // 2
            x = np.arange(w) * stride + stride // 2
            xx, yy = np.meshgrid(x, y)
            anchor_points = np.stack([xx.ravel(), yy.ravel()], axis=0)

            box_dist = pred[:4, :]
            cls_scores = pred[4, :]

            # BBox 디코딩
            x1y1 = anchor_points - box_dist[:2, :] * stride
            x2y2 = anchor_points + box_dist[2:, :] * stride
            boxes = np.concatenate([x1y1, x2y2], axis=0)

            mask = cls_scores > self.obj_thresh
            if mask.any():
                all_boxes.append(boxes[:, mask])
                all_scores.append(cls_scores[mask])
                
                valid_indices = np.where(mask)[0] + anchor_count
                kpt_indices.append(valid_indices)

            anchor_count += (h * w)

        if not all_boxes:
            return []

        boxes = np.concatenate(all_boxes, axis=1).T
        scores = np.concatenate(all_scores)
        kpt_idx = np.concatenate(kpt_indices)

        objects = []
        for i in range(len(scores)):
            # Box 원본 좌표 복원
            b = boxes[i]
            x1 = np.clip((b[0] - dw) / ratio, 0, src_w).astype(int)
            y1 = np.clip((b[1] - dh) / ratio, 0, src_h).astype(int)
            x2 = np.clip((b[2] - dw) / ratio, 0, src_w).astype(int)
            y2 = np.clip((b[3] - dh) / ratio, 0, src_h).astype(int)

            # Keypoints 원본 좌표 복원 (17, 3)
            kpts = kpt_output[:, :, kpt_idx[i]].copy()
            kpts[:, 0] = np.clip((kpts[:, 0] - dw) / ratio, 0, src_w)
            kpts[:, 1] = np.clip((kpts[:, 1] - dh) / ratio, 0, src_h)

            objects.append({
                "box": [x1, y1, x2, y2],
                "score": float(scores[i]),
                "keypoints": kpts.tolist()
            })

        return objects

    def release(self):
        self.rknn.release()


# === 실행 예시 ===
if __name__ == "__main__":

    # 1. 추론 진행
    model = YOLOv8PoseRKNN("./models/pose/yolo26n-pose-RK3588_320_i8.rknn")
    src_img = cv2.imread("./images/pose.jpg")
    objects = model.infer("./images/pose.jpg")

    # 2. 이미지 위에 관절 및 BBox 그리기
    res_img = draw_pose(src_img, objects, kpt_thresh=0.3)

    # 3. 결과 저장 및 출력
    cv2.imwrite("result.jpg", res_img)

W Query dynamic range failed. Ret code: RKNN_ERR_MODEL_INVALID. (If it is a static shape RKNN model, please ignore the above warning message.)


I RKNN: [16:51:03.246] RKNN Runtime Information, librknnrt version: 2.4.0 (b458df3b4a@2026-01-17T10:53:35)
I RKNN: [16:51:03.246] RKNN Driver Information, version: 0.9.8
I RKNN: [16:51:03.246] RKNN Model Information, version: 6, toolkit version: 2.3.2(compiler version: 2.3.2 (@2025-04-03T08:26:16)), target: RKNPU v2, target platform: rk3588, framework name: ONNX, framework layout: NCHW, model inference type: static_shape
W RKNN: [16:51:03.258] query RKNN_QUERY_INPUT_DYNAMIC_RANGE error, rknn model is static shape type, please export rknn with dynamic_shapes
